# 09 — The program graph and find_start

**Goal:** walk a Solana program's instruction↔account graph, derive an account chain in the right order, and route an intent to the right STARTING instruction — fully offline, from the bundled fixture.

This is `docs/guides/graph-engineering.md` Level 3, executable.

In [ ]:
# Offline by default. Repo + fixture paths:
from pathlib import Path
import sys

here = Path.cwd()
while not (here / "pyproject.toml").exists():
    here = here.parent
sys.path.insert(0, str(here / "src"))
FIXTURES = here / "cookbook" / "fixtures"
CORPUS_DIR = here / "data" / "corpus"
print(f"fixtures: {FIXTURES}")

## 1. Load the graph

In [ ]:
import json

graph = json.loads((FIXTURES / "program-graph-example.json").read_text())
for instruction in graph['instructions']:
    accounts = ', '.join(a['name'] for a in instruction['accounts'])
    print(f"{instruction['name']:18} -> {accounts}")

## 2. Derivation order is a graph property

The purchase instruction needs `buyer_token_account`, an ATA derived from `(buyer, mint)` — but `mint` must be READ from the `item` account, which is a PDA needing `store`, which needs `authority`. The order is not documentation; it's a dependency chain you can compute.

In [ ]:
purchase = next(i for i in graph['instructions'] if i['name'] == 'purchase')

def dependencies(account: dict) -> list[str]:
    deps = []
    for seed in account.get('seeds', []):
        if not seed.startswith("'"):   # literal seeds aren't dependencies
            deps.append(seed)
    deps.extend(account.get('derived_from', []))
    if 'read_from' in account:
        deps.append(account['read_from'])
    return deps

print('dependency edges:')
for account in purchase['accounts']:
    for dep in dependencies(account):
        print(f"  {account['name']} <- {dep}")

print('\nthe order the surface computed:')
for step in purchase['derivation_order']:
    print(f'  {step}')

## 3. Exercise: verify the order is valid

A derivation order is valid when every account appears only after everything it depends on. Check it mechanically.

In [ ]:
resolved = {'authority', 'buyer', 'item_index', 'token_program', 'system_program'}
order = ['store', 'item', 'mint', 'buyer_token_account']
for name in order:
    account = next(a for a in purchase['accounts'] if a['name'] == name)
    missing = [d for d in dependencies(account) if d not in resolved]
    assert not missing, f'{name} derived before its dependencies: {missing}'
    resolved.add(name)
    print(f'{name}: OK (deps satisfied)')
print('\nthe derivation order is topologically valid')

## 4. find_start: routing by the graph, not by name similarity

Given a plain-English intent, the right answer is a STARTING instruction — determined by what must exist first. A store must be initialized before an item can be listed before a purchase can land.

In [ ]:
def find_start(intent: str) -> str:
    lowered = intent.lower()
    best = (0, None)
    for entry in graph['intents']:
        keywords = entry['intent'].lower().replace('/', ' ').split()
        overlap = sum(1 for keyword in keywords if keyword in lowered)
        if overlap > best[0]:
            best = (overlap, entry['start'])
    if best[1] is None:
        return 'REFUSE: no starting instruction matches that intent'
    return best[1]

for intent in ('I want to buy a coffee', 'help me open a shop', 'add a product to sell',
               'stake my tokens'):
    print(f'{intent!r} -> {find_start(intent)}')

## 5. What the real thing adds

Gecko's `find_start` and `comprehend_program` do this over real programs from the **Orquestra catalogue**, with seeds recovered from source when the IDL omits them, overlays measured by execution, and provenance on every edge (notebook 05). Session 13 of the bootcamp runs it live against the hosted surface — and notebook 10 closes the loop with a receipt.